In [1]:
import re
from typing import Any, Dict

from guardrails import Guard, OnFailAction

from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator
)

### Product ID detection Guardrail

In [2]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")

In [3]:
PRODUCT_ID.findall("dasdasd B0B87HSZ8K asdfasfgadsf B0BVRLW1DP")

['B0B87HSZ8K', 'B0BVRLW1DP']

In [4]:
@register_validator(name="detect_product_id", data_type="string")
class DetectProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
            )

        return PassResult()

In [5]:
guard = Guard().use(DetectProductID(on_fail=OnFailAction.NOOP))

In [6]:
text = "Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more B0BRJS644Z "

In [8]:
guardrail_outcome = guard.validate(text)

/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [9]:
guardrail_outcome

ValidationOutcome[TypeVar](call_id='4645842256', raw_llm_output='Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more B0BRJS644Z ', validation_summaries=[ValidationSummary(validator_name='DetectProductID', validator_status='fail', property_path='$', failure_reason="found 1 product ids: ['B0BRJS644Z']", error_spans=None)], validated_output='Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more B0BRJS644Z ', reask=None, validation_passed=False, error=None)

In [10]:
for log in guard.history.last.failed_validations:
    print("Exception: ", log.validation_result.error_message)

Exception:  found 1 product ids: ['B0BRJS644Z']


In [11]:
clean_text = "Some random text"

In [12]:
guardrail_outcome_clean = guard.validate(clean_text)

In [13]:
guardrail_outcome_clean = guard.validate(clean_text)

### Product ID Redaction Guardrail

In [14]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")
REPLACEMENT = "[REDACTED..]"

In [15]:
PRODUCT_ID.sub(REPLACEMENT, "dasdasd B09KQP2H7N asdfasfgadsf B0CC4HBS85")

'dasdasd [REDACTED..] asdfasfgadsf [REDACTED..]'

In [16]:
@register_validator(name="redact_product_id", data_type="string")
class RedactProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
                fix_value=PRODUCT_ID.sub(REPLACEMENT, value)
            )

        return PassResult()

In [17]:
guard_redact = Guard().use(RedactProductID(on_fail=OnFailAction.FIX))

In [18]:
outcome_redact = guard_redact.validate(text)

In [19]:
outcome_redact

ValidationOutcome[TypeVar](call_id='4669782480', raw_llm_output='Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more B0BRJS644Z ', validation_summaries=[ValidationSummary(validator_name='RedactProductID', validator_status='fail', property_path='$', failure_reason="found 1 product ids: ['B0BRJS644Z']", error_spans=None)], validated_output='Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more [REDACTED..] ', reask=None, validation_passed=True, error=None)

In [20]:
print(f"Output: {outcome_redact.validated_output}")

Output: Added 2 units of the Dirrelo case for iPad Pro 11 inch (4th/3rd/2nd Gen) to your cart. Add one more [REDACTED..] 
